In [ ]:
import os
import cv2
from tqdm import tqdm

# Paths
split_root = "dataset_split"
processed_root = "dataset_processed"
os.makedirs(processed_root, exist_ok=True)

classes = ['wood', 'glass', 'plastic', 'metal']
splits = ['train','val','test']

# Preprocessing images (only if not already processed)
for split in splits:
    for cls in classes:
        input_folder = os.path.join(split_root, split, cls)
        output_folder = os.path.join(processed_root, split, cls)
        os.makedirs(output_folder, exist_ok=True)

        # Skip if already processed
        if len(os.listdir(output_folder)) > 0:
            print(f"{split}/{cls} already processed, skipping.")
            continue

        print(f"Processing {split}/{cls} ...")
        for img_name in tqdm(os.listdir(input_folder)):
            img_path = os.path.join(input_folder, img_name)
            output_path = os.path.join(output_folder, img_name)

            img = cv2.imread(img_path)
            if img is None:
                continue

            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = cv2.resize(img, (224, 224))
            cv2.imwrite(output_path, cv2.cvtColor(img, cv2.COLOR_RGB2BGR))

print("✅ Preprocessing done (images processed only once)")



In [ ]:
from torchvision import transforms, datasets
from torch.utils.data import DataLoader
import os

# Paths
data_dir = "dataset_processed"

# Augmentation for training
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.8,1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(30),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.RandomGrayscale(p=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

# Validation/Test: only resize + normalization
val_test_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

# Datasets
train_dataset = datasets.ImageFolder(os.path.join(data_dir,"train"), transform=train_transform)
val_dataset   = datasets.ImageFolder(os.path.join(data_dir,"val"), transform=val_test_transform)
test_dataset  = datasets.ImageFolder(os.path.join(data_dir,"test"), transform=val_test_transform)

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=2)

classes = train_dataset.classes
print("✅ DataLoaders ready with augmentation")

In [ ]:
from torchvision import transforms, datasets
from torch.utils.data import DataLoader

# Data augmentation for training
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],[0.229, 0.224, 0.225])
])

# Validation and Test (no augmentation, just normalization)
val_test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],[0.229, 0.224, 0.225])
])

data_dir = "dataset_processed"

train_dataset = datasets.ImageFolder(os.path.join(data_dir, "train"), transform=train_transform)
val_dataset   = datasets.ImageFolder(os.path.join(data_dir, "val"), transform=val_test_transform)
test_dataset  = datasets.ImageFolder(os.path.join(data_dir, "test"), transform=val_test_transform)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_dataset, batch_size=16, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_dataset, batch_size=16, shuffle=False, num_workers=2)

print("✅ DataLoaders ready with augmentation and normalization")